# Who Ruled Formula 1? A 75-Year Data Story

## Imports

In [1]:
import pandas as pd
import sqlite3 
import numpy as np

In [2]:
conn = sqlite3.connect("db/formula1.db")

## 1. The Big Picture

In [24]:
# How has Formula 1 grown over the decades?

query = """ 
WITH decade_stats AS (
        SELECT 
        (year / 10) * 10 AS decade,
        COUNT(DISTINCT ra.raceId) AS total_races,
        COUNT(DISTINCT ra.circuitId) AS unique_circuits,
        COUNT(DISTINCT re.driverId) AS unique_drivers,
        COUNT(DISTINCT re.constructorId) AS unique_constructors
        FROM races ra
        JOIN results re ON ra.raceId = re.raceId
        GROUP BY (year / 10) * 10
)
        SELECT
        decade,
        total_races,
        unique_circuits,
        unique_drivers,
        unique_constructors,
        COALESCE( total_races - LAG(total_races) OVER (ORDER BY decade), 0) AS races_growth
        FROM decade_stats
        ORDER BY decade
        """

pd.read_sql(query, conn)

,decade,total_races,unique_circuits,unique_drivers,unique_constructors,races_growth
0,1950,84,19,332,69,0
1,1960,100,25,219,71,16
2,1970,144,28,173,55,44
3,1980,156,31,115,31,12
4,1990,162,26,105,33,6
5,2000,174,24,71,23,12
6,2010,198,27,66,19,24
7,2020,107,30,36,14,-91


In [28]:
# Which decades had the most races in Formula 1 history?

query = """ 
        SELECT 
        (year / 10) * 10 AS decade,
        RANK() OVER(ORDER BY COUNT(DISTINCT ra.raceId) DESC) as busiest_decade_rank,
        COUNT(DISTINCT ra.raceId) AS total_races
        FROM races ra
        GROUP BY (year / 10) * 10
        ORDER BY decade
        """

pd.read_sql(query, conn)

,decade,busiest_decade_rank,total_races
0,1950,8,84
1,1960,7,100
2,1970,5,144
3,1980,4,156
4,1990,3,162
5,2000,2,174
6,2010,1,198
7,2020,6,107


In [27]:
# Quick summary statistics
query = """
    SELECT
        COUNT(DISTINCT raceId) AS total_races,
        COUNT(DISTINCT circuitId) AS total_circuits,
        MIN(year) AS first_season,
        MAX(year) AS last_season
    FROM races
"""

pd.read_sql(query, conn)

,total_races,total_circuits,first_season,last_season
0,1125,77,1950,2024


## 2. Constructor Dynasties

In [ ]:
#How many race entries have the most popular constructors made in F1 history?

query = """ 
with constructor_stats as(
        SELECT 
        c.name,
        count(*) as total_entries
        FROM constructors c
        INNER JOIN results re on re.constructorId=c.constructorId
        GROUP BY c.constructorId
)
        SELECT 
        RANK() OVER(ORDER BY total_entries DESC) as entry_rank,
        name,
        total_entries
        FROM constructor_stats
        LIMIT 10
        """

pd.read_sql(query, conn)

,entry_rank,name,total_entries
0,1,Ferrari,2439
1,2,McLaren,1923
2,3,Williams,1676
3,4,Tyrrell,881
4,5,Team Lotus,871
5,6,Sauber,837
6,7,Red Bull,788
7,8,Renault,787
8,9,Minardi,672
9,10,Brabham,662


In [66]:
# Which constructors won the most races?

query = """ 
WITH constructor_wins AS (
        SELECT 
        (ra.year / 10) * 10 AS decade,
        c.name,
        COUNT(*) AS total_wins
        FROM constructors c
        INNER JOIN results re ON re.constructorId = c.constructorId
        INNER JOIN races ra ON re.raceId = ra.raceId
        WHERE re.position = 1
        GROUP BY c.constructorId, decade
),
ranked AS (
        SELECT 
        RANK() OVER(ORDER BY total_wins DESC) AS win_rank,
        name,
        total_wins
        FROM constructor_wins
)
        SELECT *
        FROM ranked
        WHERE win_rank <= 3
        ORDER BY win_rank
"""

pd.read_sql(query, conn)

,win_rank,name,total_wins
0,1,Mercedes,93
1,2,Ferrari,85
2,3,Williams,61


In [ ]:
# Which constructors dominated (in terms of total_wins) each decade of Formula 1?

query = """ 
WITH constructor_wins AS (
        SELECT 
        (ra.year / 10) * 10 AS decade,
        c.name,
        COUNT(*) AS total_wins
        FROM constructors c
        INNER JOIN results re ON re.constructorId = c.constructorId
        INNER JOIN races ra ON re.raceId = ra.raceId
        WHERE re.position = 1
        GROUP BY c.constructorId, decade
),
ranked AS (
        SELECT 
        decade,
        RANK() OVER(PARTITION BY decade ORDER BY total_wins DESC) AS win_rank,
        name,
        total_wins
        FROM constructor_wins
)
        SELECT *
        FROM ranked
        WHERE win_rank <= 3
        ORDER BY decade, win_rank
"""

pd.read_sql(query, conn)

,decade,win_rank,name,total_wins
0,1950,1,Ferrari,30
1,1950,2,Alfa Romeo,11
2,1950,3,Vanwall,10
3,1960,1,Lotus-Climax,22
4,1960,2,Ferrari,13
5,1960,3,BRM,12
6,1970,1,Ferrari,37
7,1970,2,Team Lotus,35
8,1970,3,Tyrrell,21
9,1980,1,McLaren,56


## 3. Greatest Drivers

In [161]:
# Who are the all-time top 10 drivers by points, wins and podiums? 

query = """ 
WITH drivers_result_info AS(
       SELECT 
       d.driverId,
       d.forename,
       d.surname,
       re.points,
       CASE re.position
              WHEN 1 THEN 1
              ELSE 0
       END AS win,
       CASE 
              WHEN re.position BETWEEN 1 AND 3 THEN 1
              ELSE 0
       END AS podium
       FROM drivers d 
       INNER JOIN results re ON d.driverId = re.driverId
)
       SELECT
       RANK() OVER(ORDER BY SUM(points) DESC) AS rank,
       forename || ' ' || surname AS best_driver,
       SUM(points) AS total_points,
       SUM(win) AS total_wins,
       SUM(podium) AS total_podiums
       FROM drivers_result_info
       GROUP BY driverId
       ORDER BY rank
       LIMIT 10
"""

pd.read_sql(query, conn)

,rank,best_driver,total_points,total_wins,total_podiums
0,1,Lewis Hamilton,4820.5,105,220
1,2,Sebastian Vettel,3098.0,53,173
2,3,Max Verstappen,2912.5,63,122
3,4,Fernando Alonso,2329.0,32,183
4,5,Kimi Räikkönen,1873.0,21,170
5,6,Valtteri Bottas,1788.0,10,148
6,7,Nico Rosberg,1594.5,23,99
7,8,Sergio Pérez,1585.0,6,126
8,9,Michael Schumacher,1566.0,91,176
9,10,Charles Leclerc,1363.0,8,62


In [162]:
# How did the top 5 drivers accumulate points throughout their careers?

query = """ 
WITH top_5_drivers AS(
        SELECT 
        d.driverId,
        d.forename,
        d.surname,
        RANK() OVER(ORDER BY SUM(re.points) DESC) AS rank,
        SUM(re.points) as total_points
        FROM drivers d 
        INNER JOIN results re ON d.driverId = re.driverId
        GROUP BY d.driverID
        ORDER BY total_points DESC
        LIMIT 5
), points_and_dates AS(
        SELECT
        t.forename,
        t.surname,
        t.rank,
        (ra.year / 10) * 10 AS decade,
        SUM(SUM(re.points)) OVER(PARTITION BY t.driverId ORDER BY (ra.year / 10) * 10 ASC) AS accumulated_points
        FROM top_5_drivers t 
        INNER JOIN results re ON t.driverId = re.driverId
        INNER JOIN races ra ON ra.raceId = re.raceId
        GROUP BY t.driverId, decade
)
        SELECT
        forename || ' ' || surname AS driver,
        decade,
        accumulated_points
        FROM points_and_dates
        ORDER BY rank, decade
        
"""

pd.read_sql(query, conn)

,driver,decade,accumulated_points
0,Lewis Hamilton,2000,256.0
1,Lewis Hamilton,2010,3431.0
2,Lewis Hamilton,2020,4820.5
3,Sebastian Vettel,2000,125.0
4,Sebastian Vettel,2010,2985.0
5,Sebastian Vettel,2020,3098.0
6,Max Verstappen,2010,948.0
7,Max Verstappen,2020,2912.5
8,Fernando Alonso,2000,577.0
9,Fernando Alonso,2010,1899.0


In [163]:
# Which nationalities have the most points and who were their most succesful drivers?

query = """ 
WITH best_nationalities AS(
    SELECT 
    d.nationality,
    SUM(re.points) as nationality_points
    FROM drivers d
    INNER JOIN results re ON re.driverId = d.driverId
    GROUP BY nationality
    ORDER BY nationality_points DESC
    LIMIT 10
), drivers_total_points AS(
    SELECT 
    d.driverId,
    d.forename,
    d.surname,
    d.nationality,
    SUM(points) AS driver_points
    FROM drivers d
    INNER JOIN results re ON re.driverId = d.driverId
    GROUP BY d.driverId
)
    SELECT
    b.nationality,
    b.nationality_points,
    d.forename || ' ' || d.surname as best_driver,
    d.driver_points as best_driver_total_points
    FROM best_nationalities b
    INNER JOIN drivers_total_points d ON b.nationality=d.nationality
    AND (d.driver_points,d.nationality) IN 
    (     
        SELECT
        MAX(driver_points),
        nationality
        FROM drivers_total_points
        GROUP BY nationality
    )
    ORDER BY nationality_points DESC
        """


pd.read_sql(query, conn)

,nationality,nationality_points,best_driver,best_driver_total_points
0,British,11908.64,Lewis Hamilton,4820.5
1,German,7988.50,Sebastian Vettel,3098.0
2,Finnish,4388.50,Kimi Räikkönen,1873.0
3,French,3636.33,Alain Prost,798.5
4,Spanish,3614.50,Fernando Alonso,2329.0
5,Brazilian,3423.00,Felipe Massa,1167.0
6,Australian,3188.50,Daniel Ricciardo,1320.0
7,Dutch,2941.50,Max Verstappen,2912.5
8,Italian,2040.66,Riccardo Patrese,281.0
9,Mexican,1679.00,Sergio Pérez,1585.0


## 4. Circuits

In [197]:
# Which circuits are the most and the least dangerous? (R= Retired)
# We ignore circuits with less than a two hundred entries where DNF can be unstable

query = """ 
WITH circuit_retired_stats AS(
    SELECT 
    c.circuitId,
    c.circuitRef,
    c.name,
    COUNT(*) FILTER(WHERE re.positionText='R') AS retired,
    COUNT(*) AS all_entries
    FROM circuits c
    INNER JOIN races ra ON ra.circuitId = c.circuitId
    INNER JOIN results re ON re.raceId = ra.raceId
    GROUP BY c.circuitId
    HAVING COUNT(*) >= 200
)
    SELECT * FROM (  
        SELECT
        circuitRef,
        name,
        'most dangerous' as category,
        ROUND(retired * 100.0 / all_entries, 2) AS dnf_rate,
        retired,
        all_entries
        FROM circuit_retired_stats
        ORDER BY dnf_rate DESC
        LIMIT 10 )
    UNION ALL
        SELECT * FROM(
        SELECT
        circuitRef,
        name,
        'least dangerous' as category,
        ROUND(retired * 100.0 / all_entries, 2) AS dnf_rate,
        retired,
        all_entries
        FROM circuit_retired_stats
        ORDER BY dnf_rate ASC
        LIMIT 10 )
    ORDER BY dnf_rate DESC
"""

pd.read_sql(query, conn)

,circuitRef,name,category,dnf_rate,retired,all_entries
0,adelaide,Adelaide Street Circuit,most dangerous,47.12,147,312
1,brands_hatch,Brands Hatch,most dangerous,45.45,170,374
2,long_beach,Long Beach,most dangerous,45.00,99,220
3,zolder,Zolder,most dangerous,43.62,123,282
4,jacarepagua,Autódromo Internacional Nelson Piquet,most dangerous,42.66,122,286
5,galvez,Autódromo Juan y Oscar Gálvez,most dangerous,41.96,188,448
6,kyalami,Kyalami,most dangerous,41.90,212,506
7,reims,Reims-Gueux,most dangerous,40.94,104,254
8,indianapolis,Indianapolis Motor Speedway,most dangerous,40.84,234,573
9,watkins_glen,Watkins Glen,most dangerous,40.21,195,485


In [200]:
# Which circuits produce the most and the least overtakes on average?
# We ignore circuits with less than two hundred entries where results can be unstable

query = """ 
WITH circuit_overtake_stats AS(
    SELECT 
    c.circuitId,
    c.circuitRef,
    c.name,
    ROUND(AVG(ABS(re.grid - re.positionOrder)), 2) AS avg_position_change,
    COUNT(*) AS all_entries
    FROM circuits c
    INNER JOIN races ra ON ra.circuitId = c.circuitId
    INNER JOIN results re ON re.raceId = ra.raceId
    GROUP BY c.circuitId
    HAVING COUNT(*) >= 200
)
    SELECT * FROM (  
        SELECT
        circuitRef,
        name,
        'most overtakes' AS category,
        avg_position_change,
        all_entries
        FROM circuit_overtake_stats
        ORDER BY avg_position_change DESC
        LIMIT 10 )
    UNION ALL
        SELECT * FROM(
        SELECT
        circuitRef,
        name,
        'least overtakes' AS category,
        avg_position_change,
        all_entries
        FROM circuit_overtake_stats
        ORDER BY avg_position_change ASC
        LIMIT 10 )
    ORDER BY category DESC, avg_position_change DESC
"""

pd.read_sql(query, conn)

,circuitRef,name,category,avg_position_change,all_entries
0,zolder,Zolder,most overtakes,9.97,282
1,jacarepagua,Autódromo Internacional Nelson Piquet,most overtakes,9.96,286
2,long_beach,Long Beach,most overtakes,9.50,220
3,jarama,Jarama,most overtakes,9.47,236
4,adelaide,Adelaide Street Circuit,most overtakes,9.05,312
5,jerez,Circuito de Jerez,most overtakes,8.83,204
6,estoril,Autódromo do Estoril,most overtakes,8.31,367
7,indianapolis,Indianapolis Motor Speedway,most overtakes,8.28,573
8,brands_hatch,Brands Hatch,most overtakes,8.24,374
9,ricard,Circuit Paul Ricard,most overtakes,7.95,481
